# 1.3 Autoencoders: teoria, implementacion y aplicaciones

Este notebook cubre el concepto de autoencoder desde sus fundamentos matematicos hasta implementaciones practicas en PyTorch. Se trabajan cuatro variantes: el autoencoder basico, el autoencoder convolucional, el denoising autoencoder y el variational autoencoder (VAE).

El dataset utilizado es MNIST (digitos escritos a mano), por su sencillez visual y su uso historico como banco de pruebas en este tipo de modelos.

---

## Contenidos

1. Fundamentos matematicos
2. Preparacion del entorno y datos
3. Autoencoder basico (denso)
4. Autoencoder convolucional
5. Denoising Autoencoder
6. Variational Autoencoder (VAE)
7. Deteccion de anomalias con autoencoders
8. Comparativa de variantes

---

## 1. Fundamentos matematicos

Un autoencoder es una red neuronal que aprende dos funciones:

- **Encoder** $f_\theta$: comprime la entrada $x$ en una representacion latente $z$ de menor dimension.
$$z = f_\theta(x)$$

- **Decoder** $g_\phi$: reconstruye la entrada desde la representacion latente.
$$\hat{x} = g_\phi(z)$$

El entrenamiento minimiza el **error de reconstruccion** entre $x$ y $\hat{x}$:

$$\mathcal{L}(x, \hat{x}) = \|x - g_\phi(f_\theta(x))\|^2$$

No se usan etiquetas externas: la propia entrada es el objetivo. Esto lo convierte en un metodo de **aprendizaje no supervisado**.

### Intuicion geometrica

Si los datos de entrada viven en un espacio de dimension $D$ (por ejemplo, imagenes de 784 pixeles), el autoencoder aprende a proyectarlos en un espacio de dimension $d \ll D$ (el espacio latente) que captura la estructura esencial de los datos, y luego a invertir esa proyeccion.

La clave esta en el **cuello de botella**: al forzar a la informacion a pasar por una representacion comprimida, la red no puede memorizar la entrada y debe aprender caracteristicas relevantes.

---

## 2. Preparacion del entorno y datos

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# Dispositivo de computo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# Transformacion: imagen PIL -> tensor normalizado en [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

# Dataset MNIST
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# DataLoaders
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Ejemplos de entrenamiento : {len(train_dataset):,}')
print(f'Ejemplos de prueba        : {len(test_dataset):,}')
print(f'Forma de una imagen       : {train_dataset[0][0].shape}')

In [ ]:
# Visualizar ejemplos del dataset
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(16):
    img, label = train_dataset[i]
    ax = axes[i // 8, i % 8]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
plt.suptitle('Ejemplos del dataset MNIST', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Autoencoder basico (capas densas)

La variante mas simple. El encoder y el decoder estan compuestos por capas lineales (fully connected). La imagen de 28x28 se aplana a un vector de 784 elementos antes de entrar al encoder.

**Arquitectura:**

```
Encoder: 784 -> 256 -> 128 -> 64 -> z (32)
Decoder:  32 ->  64 -> 128 -> 256 -> 784
```

El espacio latente tiene dimension 32, es decir, la imagen se comprime en un factor de 784/32 = 24.5x.

In [ ]:
class Encoder(nn.Module):
    """Comprime la entrada de 784 dimensiones a un vector latente de dimension latent_dim."""

    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )

    def forward(self, x):
        # x: (batch, 1, 28, 28) -> aplanar -> (batch, 784)
        x = x.view(x.size(0), -1)
        return self.net(x)


class Decoder(nn.Module):
    """Reconstruye la imagen desde el vector latente."""

    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            # Sigmoid: fuerza la salida a [0,1], igual que la entrada normalizada
            nn.Linear(256, 784),
            nn.Sigmoid(),
        )

    def forward(self, z):
        out = self.net(z)
        # Reorganizar de (batch, 784) a (batch, 1, 28, 28)
        return out.view(-1, 1, 28, 28)


class Autoencoder(nn.Module):
    """Combina encoder y decoder en un unico modulo."""

    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z    = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z


# Instanciar el modelo
LATENT_DIM = 32
ae = Autoencoder(latent_dim=LATENT_DIM).to(device)

# Contar parametros
n_params = sum(p.numel() for p in ae.parameters() if p.requires_grad)
print(f'Parametros entrenables: {n_params:,}')
print(ae)

In [ ]:
def train_autoencoder(model, loader, epochs=15, lr=1e-3):
    """
    Bucle de entrenamiento generico para autoencoders.

    La funcion de perdida es MSE entre la imagen original y la reconstruida.
    No se usan etiquetas: la propia imagen es el objetivo.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history   = []

    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for imgs, _ in loader:           # Las etiquetas se ignoran
            imgs = imgs.to(device)

            # Paso hacia adelante
            xhat, _ = model(imgs)
            loss     = criterion(xhat, imgs)

            # Paso hacia atras
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)

        avg_loss = total_loss / len(loader.dataset)
        history.append(avg_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoca {epoch:3d}/{epochs}  |  Perdida MSE: {avg_loss:.6f}')

    return history


print('Entrenando el autoencoder basico...')
history_ae = train_autoencoder(ae, train_loader, epochs=20)

In [ ]:
# Curva de perdida
plt.figure(figsize=(8, 4))
plt.plot(history_ae, color='steelblue', linewidth=2)
plt.xlabel('Epoca')
plt.ylabel('MSE promedio')
plt.title('Curva de entrenamiento - Autoencoder basico')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def visualize_reconstructions(model, loader, n=8, title='Reconstrucciones'):
    """
    Muestra imagenes originales (fila superior) y sus reconstrucciones (fila inferior).
    Permite evaluar visualmente la calidad del autoencoder.
    """
    model.eval()
    imgs, _ = next(iter(loader))
    imgs = imgs[:n].to(device)

    with torch.no_grad():
        recon, _ = model(imgs)

    imgs  = imgs.cpu()
    recon = recon.cpu()

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(imgs[i].squeeze(), cmap='gray')
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=10)

        axes[1, i].imshow(recon[i].squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstruida', fontsize=10)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


visualize_reconstructions(ae, test_loader, title='Autoencoder basico - Test set')

In [ ]:
def visualize_latent_space(model, loader, n_samples=3000, title='Espacio latente'):
    """
    Proyecta el espacio latente en 2D usando PCA y colorea por digito.
    Permite ver si el modelo agrupa correctamente los digitos en el espacio latente,
    aunque no fue entrenado con etiquetas.
    """
    model.eval()
    latents, labels = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            _, z = model(imgs)
            latents.append(z.cpu().numpy())
            labels.append(lbls.numpy())
            if sum(len(l) for l in labels) >= n_samples:
                break

    latents = np.concatenate(latents)[:n_samples]
    labels  = np.concatenate(labels)[:n_samples]

    # Reduccion a 2D con PCA
    pca  = PCA(n_components=2)
    z_2d = pca.fit_transform(latents)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=labels, cmap='tab10',
                          s=5, alpha=0.7)
    plt.colorbar(scatter, label='Digito')
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
    plt.title(title)
    plt.tight_layout()
    plt.show()


visualize_latent_space(ae, test_loader, title='Espacio latente - Autoencoder basico (PCA 2D)')

### Observacion

A pesar de que el modelo no fue entrenado con etiquetas, el espacio latente muestra una organizacion que refleja la estructura real de los datos: digitos similares tienden a agruparse en regiones proximas. Esto demuestra que el autoencoder aprende caracteristicas semanticamente significativas de forma no supervisada.

---

## 4. Autoencoder Convolucional

Las capas densas tratan cada pixel de forma independiente, ignorando la estructura espacial de la imagen. Las **capas convolucionales** comparten pesos y explotan la correlacion local entre pixeles vecinos, lo que las hace mas adecuadas para datos visuales.

**Arquitectura:**

```
Encoder:
  (1, 28, 28) -> Conv(32, 3) -> ReLU -> MaxPool -> (32, 14, 14)
  (32, 14, 14) -> Conv(64, 3) -> ReLU -> MaxPool -> (64, 7, 7)
  Flatten -> Linear(64*7*7, latent_dim)

Decoder:
  Linear(latent_dim, 64*7*7) -> Reshape
  ConvTranspose(64, 3, stride=2) -> ReLU -> (32, 14, 14)
  ConvTranspose(32, 3, stride=2) -> ReLU -> (1, 28, 28)
  Sigmoid
```

La `ConvTranspose2d` (convolucion transpuesta) es la operacion inversa al pooling: aumenta la resolucion espacial aplicando una convolucion con stride.

In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        # Bloque convolucional 1: extrae caracteristicas de bajo nivel
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (1,28,28) -> (32,28,28)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (32,28,28) -> (32,14,14)
        )
        # Bloque convolucional 2: extrae caracteristicas de alto nivel
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (32,14,14) -> (64,14,14)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (64,14,14) -> (64, 7, 7)
        )
        # Proyeccion al espacio latente
        self.fc = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        x = self.conv1(x)              # (B, 32, 14, 14)
        x = self.conv2(x)              # (B, 64,  7,  7)
        x = x.view(x.size(0), -1)     # (B, 64*7*7 = 3136)
        z = self.fc(x)                 # (B, latent_dim)
        return z


class ConvDecoder(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        # Proyeccion desde el espacio latente al mapa de caracteristicas inicial
        self.fc = nn.Linear(latent_dim, 64 * 7 * 7)

        # ConvTranspose2d: convolucion transpuesta para aumentar resolucion
        # stride=2 duplica el tamano espacial en cada bloque
        self.deconv1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2,
                               padding=1, output_padding=1),  # (64,7,7) -> (32,14,14)
            nn.ReLU(),
        )
        self.deconv2 = nn.Sequential(
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2,
                               padding=1, output_padding=1),  # (32,14,14) -> (1,28,28)
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)                       # (B, 3136)
        x = x.view(-1, 64, 7, 7)            # (B, 64,  7,  7)
        x = self.deconv1(x)                  # (B, 32, 14, 14)
        x = self.deconv2(x)                  # (B,  1, 28, 28)
        return x


class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.encoder = ConvEncoder(latent_dim)
        self.decoder = ConvDecoder(latent_dim)

    def forward(self, x):
        z    = self.encoder(x)
        xhat = self.decoder(z)
        return xhat, z


conv_ae = ConvAutoencoder(latent_dim=LATENT_DIM).to(device)
n_params = sum(p.numel() for p in conv_ae.parameters() if p.requires_grad)
print(f'Parametros entrenables (Conv AE): {n_params:,}')

In [ ]:
print('Entrenando el autoencoder convolucional...')
history_conv = train_autoencoder(conv_ae, train_loader, epochs=20)

# Comparativa de curvas de perdida
plt.figure(figsize=(8, 4))
plt.plot(history_ae,   label='Basico (denso)',   color='steelblue', linewidth=2)
plt.plot(history_conv, label='Convolucional',    color='darkorange', linewidth=2)
plt.xlabel('Epoca')
plt.ylabel('MSE promedio')
plt.title('Comparativa de perdida de entrenamiento')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
visualize_reconstructions(conv_ae, test_loader, title='Autoencoder convolucional - Test set')
visualize_latent_space(conv_ae, test_loader, title='Espacio latente - Autoencoder convolucional (PCA 2D)')

---

## 5. Denoising Autoencoder (DAE)

La idea central del **denoising autoencoder** es entrenar el modelo para que aprenda a eliminar el ruido de las entradas. En lugar de pasarle la imagen limpia como entrada, se le pasa una version **corrompida** con ruido gaussiano, y la red debe reconstruir la imagen original sin ruido.

$$\hat{x} = g_\phi(f_\theta(\tilde{x})), \quad \tilde{x} = x + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

Esto fuerza al modelo a aprender representaciones **robustas**: no puede memorizar los detalles superficiales del ruido, sino que debe capturar la estructura global del digito.

**Aplicaciones directas:**
- Restauracion de imagenes medicas con ruido de adquisicion.
- Recuperacion de senales de audio degradadas.
- Preprocesamiento robusto ante datos corrompidos.

In [ ]:
def add_gaussian_noise(imgs, std=0.3):
    """
    Agrega ruido gaussiano a un batch de imagenes y recorta los valores a [0,1].
    std controla la intensidad del ruido: valores tipicos entre 0.1 y 0.5.
    """
    noise  = torch.randn_like(imgs) * std
    noisy  = imgs + noise
    return torch.clamp(noisy, 0.0, 1.0)


# Visualizar el efecto del ruido
sample_imgs, _ = next(iter(test_loader))
sample_noisy   = add_gaussian_noise(sample_imgs[:8], std=0.3)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(sample_imgs[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Original', fontsize=10)

    axes[1, i].imshow(sample_noisy[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('Con ruido', fontsize=10)

plt.suptitle('Efecto del ruido gaussiano (std=0.3)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def train_denoising_ae(model, loader, epochs=20, lr=1e-3, noise_std=0.3):
    """
    Entrenamiento especifico para denoising autoencoder.

    La diferencia clave con el entrenamiento normal:
    - Entrada al modelo: imagen corrompida con ruido
    - Objetivo (target): imagen original sin ruido
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history   = []

    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for imgs, _ in loader:
            imgs      = imgs.to(device)
            noisy     = add_gaussian_noise(imgs, std=noise_std)  # entrada corrompida

            xhat, _   = model(noisy)           # reconstruccion desde la entrada ruidosa
            loss      = criterion(xhat, imgs)  # comparar con la imagen LIMPIA

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)

        avg_loss = total_loss / len(loader.dataset)
        history.append(avg_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoca {epoch:3d}/{epochs}  |  Perdida MSE: {avg_loss:.6f}')

    return history


# Reutilizamos la arquitectura convolucional para el DAE
dae = ConvAutoencoder(latent_dim=LATENT_DIM).to(device)

print('Entrenando el Denoising Autoencoder...')
history_dae = train_denoising_ae(dae, train_loader, epochs=20, noise_std=0.3)

In [ ]:
def visualize_denoising(model, loader, n=8, noise_std=0.3):
    """Muestra tres filas: original / ruidosa / reconstruida limpia."""
    model.eval()
    imgs, _ = next(iter(loader))
    imgs    = imgs[:n].to(device)
    noisy   = add_gaussian_noise(imgs, std=noise_std)

    with torch.no_grad():
        recon, _ = model(noisy)

    imgs  = imgs.cpu()
    noisy = noisy.cpu()
    recon = recon.cpu()

    fig, axes = plt.subplots(3, n, figsize=(2 * n, 6))
    etiquetas = ['Original', f'Ruidosa (std={noise_std})', 'Reconstruida']
    datos     = [imgs, noisy, recon]

    for row, (data, label) in enumerate(zip(datos, etiquetas)):
        for col in range(n):
            axes[row, col].imshow(data[col].squeeze(), cmap='gray')
            axes[row, col].axis('off')
        axes[row, 0].set_ylabel(label, fontsize=10)

    plt.suptitle('Denoising Autoencoder', fontsize=13)
    plt.tight_layout()
    plt.show()


visualize_denoising(dae, test_loader, noise_std=0.3)

---

## 6. Variational Autoencoder (VAE)

El **VAE** (Kingma & Welling, 2013) es la extension generativa del autoencoder. En lugar de mapear $x$ a un vector fijo $z$, el encoder aprende una **distribucion de probabilidad** sobre el espacio latente:

$$z \sim q_\phi(z | x) = \mathcal{N}(\mu_\phi(x),\, \sigma^2_\phi(x))$$

El decoder reconstruye desde una muestra de esa distribucion:

$$\hat{x} = g_\theta(z), \quad z \sim q_\phi(z | x)$$

### Funcion de perdida: ELBO

El VAE maximiza la **Evidence Lower BOund (ELBO)**:

$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathbb{E}[\log p_\theta(x | z)]}_{\text{reconstruccion}} - \underbrace{D_{\text{KL}}(q_\phi(z|x) \| p(z))}_{\text{regularizacion}}$$

- **Termino de reconstruccion**: el decoder debe recuperar bien la entrada (equivale al MSE o BCE).
- **Divergencia KL**: fuerza a la distribucion posterior aprendida $q_\phi(z|x)$ a parecerse a la prior $p(z) = \mathcal{N}(0, I)$. Esto regulariza el espacio latente para que sea continuo y estructurado.

### Truco de reparametrizacion

El muestreo $z \sim \mathcal{N}(\mu, \sigma^2)$ no es diferenciable, lo que impide la retropropagacion. El **truco de reparametrizacion** lo resuelve descomponiendo el muestreo en:

$$z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Ahora el gradiente fluye a traves de $\mu$ y $\sigma$ (parametros del encoder), mientras que $\epsilon$ es ruido externo.

In [ ]:
class VAEEncoder(nn.Module):
    """
    Encoder del VAE: en lugar de un vector z, produce mu y log_var.
    log_var en lugar de sigma directamente por estabilidad numerica.
    """
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(784, 400),
            nn.ReLU(),
            nn.Linear(400, 200),
            nn.ReLU(),
        )
        # Dos cabezas separadas: una para mu, otra para log_var
        self.fc_mu      = nn.Linear(200, latent_dim)
        self.fc_log_var = nn.Linear(200, latent_dim)

    def forward(self, x):
        x       = x.view(x.size(0), -1)
        h       = self.shared(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var


class VAEDecoder(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 200),
            nn.ReLU(),
            nn.Linear(200, 400),
            nn.ReLU(),
            nn.Linear(400, 784),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class VAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)

    def reparametrize(self, mu, log_var):
        """
        Truco de reparametrizacion: z = mu + std * epsilon
        epsilon se muestrea de N(0, I) en cada paso de entrenamiento.
        std = exp(0.5 * log_var) para garantizar positividad.
        """
        std     = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(std)     # ruido externo, sin gradiente
        return mu + std * epsilon

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z           = self.reparametrize(mu, log_var)
        xhat        = self.decoder(z)
        return xhat, mu, log_var

    def sample(self, n_samples: int, device):
        """Genera muestras nuevas muestreando z directamente de la prior N(0,I)."""
        z = torch.randn(n_samples, self.encoder.fc_mu.out_features).to(device)
        with torch.no_grad():
            return self.decoder(z)


vae = VAE(latent_dim=16).to(device)
n_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print(f'Parametros entrenables (VAE): {n_params:,}')

In [ ]:
def vae_loss(xhat, x, mu, log_var, beta=1.0):
    """
    Perdida del VAE = Reconstruccion + beta * KL

    Reconstruccion: Binary Cross Entropy pixel a pixel.
    KL: divergencia entre la posterior aprendida N(mu, sigma^2) y la prior N(0,1).
        Forma cerrada: -0.5 * sum(1 + log_var - mu^2 - exp(log_var))

    beta=1 es el VAE original. beta>1 promueve disentanglement (beta-VAE).
    """
    # BCE suma sobre pixeles y promedia sobre el batch
    recon_loss = F.binary_cross_entropy(xhat, x, reduction='sum') / x.size(0)

    # KL divergencia (forma analitica cerrada para distribuciones gaussianas)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)

    return recon_loss + beta * kl_loss, recon_loss, kl_loss


def train_vae(model, loader, epochs=25, lr=1e-3, beta=1.0):
    optimizer       = optim.Adam(model.parameters(), lr=lr)
    history         = {'total': [], 'recon': [], 'kl': []}

    model.train()
    for epoch in range(1, epochs + 1):
        tot_loss = tot_recon = tot_kl = 0.0

        for imgs, _ in loader:
            imgs         = imgs.to(device)
            xhat, mu, lv = model(imgs)

            loss, recon, kl = vae_loss(xhat, imgs, mu, lv, beta=beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tot_loss  += loss.item()  * imgs.size(0)
            tot_recon += recon.item() * imgs.size(0)
            tot_kl    += kl.item()   * imgs.size(0)

        n = len(loader.dataset)
        history['total'].append(tot_loss  / n)
        history['recon'].append(tot_recon / n)
        history['kl'].append(tot_kl    / n)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoca {epoch:3d}/{epochs}  |  '
                  f'Total: {tot_loss/n:.2f}  '
                  f'Recon: {tot_recon/n:.2f}  '
                  f'KL: {tot_kl/n:.2f}')

    return history


print('Entrenando el VAE...')
history_vae = train_vae(vae, train_loader, epochs=30, beta=1.0)

In [ ]:
# Curvas de perdida del VAE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key, color in zip(axes, ['total', 'recon', 'kl'],
                           ['steelblue', 'darkorange', 'seagreen']):
    ax.plot(history_vae[key], color=color, linewidth=2)
    ax.set_title(f'Perdida: {key}', fontsize=12)
    ax.set_xlabel('Epoca')
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas de entrenamiento - VAE', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Reconstrucciones del VAE
vae.eval()
imgs, _ = next(iter(test_loader))
imgs    = imgs[:8].to(device)

with torch.no_grad():
    xhat, _, _ = vae(imgs)

imgs = imgs.cpu()
xhat = xhat.cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(imgs[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(xhat[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Reconstruida', fontsize=10)
plt.suptitle('VAE - Reconstrucciones en test set', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Generacion de muestras nuevas desde la prior N(0, I)
# Esta es la capacidad generativa exclusiva del VAE
vae.eval()
generated = vae.sample(n_samples=16, device=device).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    axes[i // 8, i % 8].imshow(generated[i].squeeze(), cmap='gray')
    axes[i // 8, i % 8].axis('off')

plt.suptitle('VAE - Muestras generadas desde z ~ N(0,I)\n'
             '(sin utilizar ninguna imagen real)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def interpolate_latent(model, loader, device, steps=10):
    """
    Interpolacion lineal entre dos puntos del espacio latente.

    Dado que el espacio latente del VAE es continuo (gracias a la regularizacion KL),
    los puntos intermedios producen imagenes coherentes: la transicion es suave.
    En un autoencoder basico sin regularizacion esto no esta garantizado.
    """
    model.eval()
    imgs, _ = next(iter(loader))
    imgs    = imgs[:2].to(device)

    with torch.no_grad():
        mu, _ = model.encoder(imgs)
        z1, z2 = mu[0], mu[1]

        # Interpolar linealmente entre z1 y z2
        alphas      = torch.linspace(0, 1, steps).to(device)
        z_interp    = torch.stack([alpha * z2 + (1 - alpha) * z1 for alpha in alphas])
        recon_interp = model.decoder(z_interp).cpu()

    fig, axes = plt.subplots(1, steps, figsize=(2 * steps, 2))
    for i, ax in enumerate(axes):
        ax.imshow(recon_interp[i].squeeze(), cmap='gray')
        ax.axis('off')
        ax.set_title(f'{alphas[i].item():.1f}', fontsize=8)

    plt.suptitle('Interpolacion lineal en el espacio latente del VAE', fontsize=12)
    plt.tight_layout()
    plt.show()


interpolate_latent(vae, test_loader, device, steps=10)

---

## 7. Deteccion de anomalias con autoencoders

Esta es una de las aplicaciones mas directas e importantes de los autoencoders. La idea es simple:

1. Entrenar el autoencoder **solo con datos normales** (por ejemplo, solo el digito 0).
2. El modelo aprende a reconstruir bien los datos normales.
3. Ante una **anomalia** (cualquier otro digito), el error de reconstruccion sera alto, ya que el modelo no ha visto ese patron antes.

$$\text{score}(x) = \|x - \hat{x}\|^2 \quad \begin{cases} \text{bajo} & \Rightarrow \text{normal} \\ \text{alto} & \Rightarrow \text{anomalia} \end{cases}$$

**Aplicaciones reales:** deteccion de fraude, defectos industriales, intrusiones en redes, patologias medicas raras.

In [ ]:
from torch.utils.data import Subset

# Separar: digito 0 como clase "normal", resto como "anomalia"
NORMAL_CLASS = 0

normal_idx    = [i for i, (_, y) in enumerate(train_dataset) if y == NORMAL_CLASS]
normal_subset = Subset(train_dataset, normal_idx)
normal_loader = DataLoader(normal_subset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Ejemplos de clase normal ({NORMAL_CLASS}): {len(normal_idx):,}')
print(f'Ejemplos de entrenamiento totales        : {len(train_dataset):,}')

In [ ]:
# Entrenar el autoencoder solo con la clase normal
anomaly_ae = Autoencoder(latent_dim=16).to(device)

print(f'Entrenando solo con digito {NORMAL_CLASS}...')
history_anomaly = train_autoencoder(anomaly_ae, normal_loader, epochs=20)

In [ ]:
def compute_reconstruction_errors(model, loader, device):
    """Calcula el error de reconstruccion (MSE) por imagen en todo el loader."""
    model.eval()
    errors, labels = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs        = imgs.to(device)
            xhat, _     = model(imgs)
            # Error por imagen (promedio sobre pixeles)
            batch_errors = F.mse_loss(xhat, imgs, reduction='none')
            batch_errors = batch_errors.view(imgs.size(0), -1).mean(dim=1)

            errors.append(batch_errors.cpu().numpy())
            labels.append(lbls.numpy())

    return np.concatenate(errors), np.concatenate(labels)


errors, labels = compute_reconstruction_errors(anomaly_ae, test_loader, device)

# Separar errores por clase
normal_errors   = errors[labels == NORMAL_CLASS]
anomaly_errors  = errors[labels != NORMAL_CLASS]

print(f'Error promedio - Normal ({NORMAL_CLASS})  : {normal_errors.mean():.5f}')
print(f'Error promedio - Anomalia (resto)        : {anomaly_errors.mean():.5f}')
print(f'Razon anomalia/normal                    : {anomaly_errors.mean()/normal_errors.mean():.1f}x')

In [ ]:
# Histograma de errores de reconstruccion por clase
plt.figure(figsize=(10, 5))
plt.hist(normal_errors,  bins=80, alpha=0.6, label=f'Normal (digito {NORMAL_CLASS})',
         color='steelblue', density=True)
plt.hist(anomaly_errors, bins=80, alpha=0.6, label='Anomalia (otros digitos)',
         color='crimson', density=True)

# Umbral simple: percentil 95 de los errores normales
threshold = np.percentile(normal_errors, 95)
plt.axvline(threshold, color='black', linestyle='--', linewidth=1.5,
            label=f'Umbral (p95 normal) = {threshold:.4f}')

plt.xlabel('Error de reconstruccion (MSE)')
plt.ylabel('Densidad')
plt.title('Deteccion de anomalias: distribucion del error de reconstruccion')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Desempeno del detector con ese umbral
all_labels_binary = (labels != NORMAL_CLASS).astype(int)   # 1 = anomalia, 0 = normal
predictions       = (errors > threshold).astype(int)

tp = ((predictions == 1) & (all_labels_binary == 1)).sum()
fp = ((predictions == 1) & (all_labels_binary == 0)).sum()
fn = ((predictions == 0) & (all_labels_binary == 1)).sum()
tn = ((predictions == 0) & (all_labels_binary == 0)).sum()

precision = tp / (tp + fp + 1e-8)
recall    = tp / (tp + fn + 1e-8)
accuracy  = (tp + tn) / len(labels)

print(f'\nResultados con umbral = {threshold:.4f}')
print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'Accuracy  : {accuracy:.3f}')

In [ ]:
# Visualizar los ejemplos con mayor y menor error de reconstruccion
def show_extreme_errors(model, loader, device, n=8):
    """Muestra las imagenes con mayor y menor error de reconstruccion."""
    model.eval()
    all_imgs, all_errors, all_labels = [], [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs        = imgs.to(device)
            xhat, _     = model(imgs)
            errs        = F.mse_loss(xhat, imgs, reduction='none')
            errs        = errs.view(imgs.size(0), -1).mean(dim=1)
            all_imgs.append(imgs.cpu())
            all_errors.append(errs.cpu())
            all_labels.append(lbls)

    all_imgs   = torch.cat(all_imgs)
    all_errors = torch.cat(all_errors)
    all_labels = torch.cat(all_labels)

    # Ordenar por error
    sorted_idx = all_errors.argsort()
    lowest_idx  = sorted_idx[:n]
    highest_idx = sorted_idx[-n:].flip(0)

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 5))
    for i in range(n):
        for row, idx_list, title_prefix in [
            (0, lowest_idx,  'Bajo'),
            (1, highest_idx, 'Alto')
        ]:
            idx = idx_list[i].item()
            axes[row, i].imshow(all_imgs[idx].squeeze(), cmap='gray')
            axes[row, i].set_title(
                f'Dig:{all_labels[idx].item()}\n{all_errors[idx]:.4f}',
                fontsize=8
            )
            axes[row, i].axis('off')

    axes[0, 0].set_ylabel('Error bajo\n(normales)', fontsize=9)
    axes[1, 0].set_ylabel('Error alto\n(anomalias)', fontsize=9)
    plt.suptitle('Imagenes con menor y mayor error de reconstruccion', fontsize=13)
    plt.tight_layout()
    plt.show()


show_extreme_errors(anomaly_ae, test_loader, device)

---

## 8. Comparativa de variantes

Evaluamos las tres variantes entrenadas (basico, convolucional, denoising) en el conjunto de test con la metrica MSE de reconstruccion.

In [ ]:
def evaluate_mse(model, loader, device, noisy=False, noise_std=0.3):
    """Calcula el MSE de reconstruccion sobre todo el loader."""
    model.eval()
    total_loss = 0.0
    criterion  = nn.MSELoss()

    with torch.no_grad():
        for imgs, _ in loader:
            imgs  = imgs.to(device)
            inp   = add_gaussian_noise(imgs, noise_std) if noisy else imgs

            if isinstance(model, VAE):
                xhat, _, _ = model(inp)
            else:
                xhat, _    = model(inp)

            total_loss += criterion(xhat, imgs).item() * imgs.size(0)

    return total_loss / len(loader.dataset)


results = {
    'AE Basico (denso)'      : evaluate_mse(ae,       test_loader, device),
    'AE Convolucional'       : evaluate_mse(conv_ae,  test_loader, device),
    'Denoising AE (noisy in)': evaluate_mse(dae,      test_loader, device, noisy=True),
    'VAE'                    : evaluate_mse(vae,       test_loader, device),
}

print('MSE de reconstruccion en el conjunto de test')
print('-' * 45)
for name, mse in results.items():
    print(f'{name:<30} {mse:.6f}')

In [ ]:
# Grafica comparativa
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(list(results.keys()), list(results.values()),
               color=['steelblue', 'darkorange', 'seagreen', 'mediumpurple'])
ax.bar_label(bars, fmt='%.5f', padding=4, fontsize=10)
ax.set_xlabel('MSE de reconstruccion (menor es mejor)')
ax.set_title('Comparativa de variantes de autoencoder en MNIST')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen de variantes, caracteristicas y aplicaciones
resumen = [
    ('AE Basico',        'Capas densas',          'No',   'No',   'Reduccion dimensionalidad'),
    ('AE Convolucional', 'Conv + ConvTranspose',  'No',   'No',   'Extraccion de caracteristicas visuales'),
    ('Denoising AE',     'Cualquiera',            'Si',   'No',   'Restauracion de imagenes, robustez'),
    ('VAE',              'Capas densas + latente  prob.', 'No', 'Si', 'Generacion, interpolacion, síntesis'),
    ('beta-VAE',         'Como VAE',              'No',   'Si',   'Disentanglement, control de factores'),
    ('Sparse AE',        'Capas densas + penalty', 'No',  'No',   'Interpretabilidad, diccionarios'),
]

print(f'{"Variante":<20} {"Arquitectura":<35} {"Ruido":<8} {"Generativo":<12} {"Aplicacion principal"}')
print('-' * 110)
for row in resumen:
    print(f'{row[0]:<20} {row[1]:<35} {row[2]:<8} {row[3]:<12} {row[4]}')

---

## Conclusiones

A lo largo de este notebook se construyeron e implementaron cuatro variantes de autoencoder sobre el mismo dataset, lo que permite comparar directamente sus comportamientos:

**Autoencoder basico**: la variante mas simple. Comprime eficientemente los datos y produce representaciones latentes con estructura semantica visible, a pesar de no usar etiquetas. Sirve como linea base.

**Autoencoder convolucional**: aprovecha la estructura espacial de las imagenes mediante filtros convolucionales compartidos. Generalmente produce reconstrucciones mas nitidas con menos parametros que la version densa.

**Denoising Autoencoder**: al entrenar con entradas corrompidas y objetivos limpios, aprende representaciones mas robustas. El espacio latente captura informacion estructural en lugar de artefactos superficiales del ruido.

**Variational Autoencoder**: la extension generativa. El truco de reparametrizacion y la regularizacion KL producen un espacio latente continuo y estructurado que permite generar muestras nuevas e interpolar suavemente entre ejemplos.

La **deteccion de anomalias** ilustra una aplicacion practica directa: un modelo entrenado solo con datos normales produce errores de reconstruccion significativamente mayores ante datos fuera de la distribucion de entrenamiento.

---

## Referencias

- Hinton, G. E., & Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*, 313(5786), 504-507.
- Vincent, P., et al. (2010). Stacked denoising autoencoders. *Journal of Machine Learning Research*, 11, 3371-3408.
- Kingma, D. P., & Welling, M. (2013). Auto-encoding variational bayes. *arXiv:1312.6114*.
- Higgins, I., et al. (2017). beta-VAE: Learning basic visual concepts with a constrained variational framework. *ICLR 2017*.
- He, K., et al. (2022). Masked autoencoders are scalable vision learners. *CVPR 2022*.